In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from src.models import whiff
from src.models.whiff import build_features, fit_boosting

wm = whiff.build()
gb = fit_boosting(wm.X_train, wm.y_train)
print("iterations:", gb.n_iter_)

iterations: 292


In [2]:
def ece(y, p, bins=10):
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(p, edges[1:-1])
    return sum(
        (idx == b).sum() * abs(y[idx == b].mean() - p[idx == b].mean())
        for b in range(bins) if (idx == b).sum() > 0
    ) / len(y)

test = wm.split.test.copy()
y_test = test["target"].to_numpy()
X_test = build_features(test, wm.keep_types, columns=wm.X_train.columns)

p_gb = gb.predict_proba(X_test)[:, 1]
p_base = wm.predict_baseline(test)
p_const = np.full(len(test), wm.split.train["target"].mean())

rows = []
for name, p in [("constant", p_const), ("lookup baseline", p_base), ("boosting", p_gb)]:
    rows.append({
        "model": name,
        "log_loss": log_loss(y_test, p),
        "brier": brier_score_loss(y_test, p),
        "auc": roc_auc_score(y_test, p) if len(np.unique(p)) > 1 else np.nan,
        "ece": ece(y_test, p),
    })

test_results = pd.DataFrame(rows)
print(f"test rows: {len(test):,}  whiff rate: {y_test.mean():.4f}")
print(test_results.round(5).to_string(index=False))

test rows: 54,137  whiff rate: 0.2413
          model  log_loss   brier     auc     ece
       constant   0.55294 0.18321     NaN 0.01121
lookup baseline   0.49466 0.16031 0.71292 0.01382
       boosting   0.44690 0.14225 0.77741 0.00954


In [3]:
def reliability(y, p, bins=10):
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(p, edges[1:-1])
    rows = []
    for b in range(bins):
        m = idx == b
        if m.sum() == 0:
            continue
        rows.append({"predicted": p[m].mean(), "actual": y[m].mean(),
                     "gap": y[m].mean() - p[m].mean(), "n": int(m.sum())})
    return pd.DataFrame(rows)

print(reliability(y_test, p_gb).round(4).to_string(index=False))

 predicted  actual     gap    n
    0.0564  0.0515 -0.0048 5414
    0.0805  0.0730 -0.0076 5414
    0.1014  0.1027  0.0013 5413
    0.1243  0.1274  0.0031 5414
    0.1518  0.1600  0.0082 5413
    0.1857  0.1949  0.0092 5414
    0.2305  0.2294 -0.0011 5414
    0.2953  0.3200  0.0247 5413
    0.4049  0.4296  0.0248 5414
    0.7139  0.7246  0.0107 5414


In [4]:
ROOT = Path.cwd().parents[1]
out = ROOT / "models" / "artifacts" / "whiff_boosting" / "v1"
out.mkdir(parents=True, exist_ok=True)

joblib.dump({"model": gb, "keep_types": sorted(wm.keep_types),
             "columns": list(wm.X_train.columns)}, out / "model.joblib")

test_results.to_csv(out / "test_metrics.csv", index=False)

meta = {
    "version": "v1",
    "target": "P(whiff | swing)",
    "estimator": "HistGradientBoostingClassifier",
    "n_iterations": int(gb.n_iter_),
    "train": wm.split.summary()["train"],
    "validation": wm.split.summary()["validation"],
    "test": wm.split.summary()["test"],
    "features": list(wm.X_train.columns),
    "kept_pitch_types": sorted(wm.keep_types),
    "calibrated": False,
    "seed": 42,
}
(out / "metadata.json").write_text(json.dumps(meta, indent=2))
print(f"saved to {out}")
print(f"model size: {(out / 'model.joblib').stat().st_size / 1e6:.2f} MB")

saved to /Users/minjong/Projects/mlb-intelligence-lab/models/artifacts/whiff_boosting/v1
model size: 1.08 MB
